# Provably-safe policy optimisation: projected gradient descent

This tutorial shows how to train Stable-Baselines3 agents whose network
parameters are **projected, after every optimizer step, onto a union of
per-parameter boxes** (a *Rashomon set*). Constraining where the parameters may
move keeps the policy provably inside a certified region while it adapts to a
new task.

Everything lives in the project-wide package
[`provably_safe_policy_optimisation`](../core/provably_safe_policy_optimisation),
so it can be imported anywhere in the project:

| Object | Role |
|---|---|
| `ProjectedAdam` | Adam variant that projects (a subset of) its parameters each step |
| `ProjectedDQN` | SB3 `DQN` that projects the online `q_net` |
| `ProjectedPPO` | SB3 `PPO` that projects the **actor** branch (critic stays free) |
| `projection_target_parameter_names` | resolve the ordered actor params, to align bounds |
| `project_to_interval_union`, `validate_and_prepare_param_interval_bounds` | low-level primitives |

**Bounds are caller-supplied** and must align (count / shape / order) with the
parameters they constrain. They can be a single box (`list[Tensor]`) or a union
of boxes (`list[list[Tensor]]`). In practice they come either from an
`IntervalTrainer` Rashomon certificate or from a simple trust-region box around
the source weights (used below for clarity).

In [1]:
import gymnasium as gym
import torch as th

from provably_safe_policy_optimisation import (
    ProjectedAdam,
    ProjectedDQN,
    ProjectedPPO,
    projection_target_parameter_names,
    project_to_interval_union,
)

th.manual_seed(0)

## 1. `ProjectedAdam` — the projection as an optimizer

`ProjectedAdam` is a drop-in `torch.optim.Adam`. Until you call `set_bounds`, it
behaves exactly like Adam (projection is a no-op), so it is always safe to pass
as `optimizer_class`. Once bounds are attached, every `.step()` clamps the
parameters back into the box.

Below we give a parameter a zero gradient so Adam leaves it untouched and we can
see *only* the projection: the point `[5, -5]` is snapped into the box `[-1, 1]`.

In [2]:
w = th.nn.Parameter(th.tensor([5.0, -5.0]))
opt = ProjectedAdam([w], lr=1e-3)
opt.set_bounds([th.tensor([-1.0, -1.0])], [th.tensor([1.0, 1.0])])

w.grad = th.zeros_like(w)   # zero grad -> Adam no-op, only projection acts
opt.step()
print("projected parameter:", w.detach().tolist())   # -> [1.0, -1.0]

projected parameter: [1.0, -1.0]


## 2. `ProjectedDQN` — constrain the Q-network

`ProjectedDQN` forces `optimizer_class=ProjectedAdam` and, once the policy is
built, attaches your bounds to the **online `q_net`** (the target network is just
a periodic copy, so it needs no handling).

Bounds must be aligned to `model.policy.q_net.parameters()`. A convenient choice
is a **trust region**: a box of fixed half-width around the initial weights. We
build a probe model to read the initial weights, define the box, then train a
second model (same seed) with the bounds attached.

In [3]:
def make_dqn(**extra):
    return ProjectedDQN(
        "MlpPolicy", gym.make("CartPole-v1"), seed=0, device="cpu", verbose=0,
        learning_starts=50, buffer_size=1_000, batch_size=16,
        train_freq=1, target_update_interval=10,
        policy_kwargs={"net_arch": [16]}, **extra,
    )

# Probe: read the initial q_net weights to centre the trust region on them.
probe = make_dqn()
init = [p.detach().clone() for p in probe.policy.q_net.parameters()]
lower = [p - 0.02 for p in init]
upper = [p + 0.02 for p in init]

dqn = make_dqn(learning_rate=1e-2, param_bounds_l=lower, param_bounds_u=upper)
dqn.learn(total_timesteps=600)

inside = all(th.all(p >= lo - 1e-6) and th.all(p <= hi + 1e-6)
             for p, lo, hi in zip(dqn.policy.q_net.parameters(), lower, upper))
print("all q_net params inside the box:", inside)
print("projections fired on", dqn.policy.optimizer._projected_elements, "elements")

all q_net params inside the box: True
projections fired on 56994 elements


## 3. `ProjectedPPO` — constrain the actor, leave the critic free

SB3 PPO uses a **single optimizer over the whole policy** (actor *and* critic).
Retention bounds usually concern only the policy, so `ProjectedPPO` projects just
the **actor branch** by default (`projection_target="feature_actor"` = actor-side
features extractor + `mlp_extractor.policy_net` + `action_net`). The value network
keeps training unconstrained. (Pass `projection_target="all"` or an explicit
`projection_param_names` list to change this.)

Use `projection_target_parameter_names(model)` to get the exact ordered names so
you can build aligned bounds.

In [4]:
def make_ppo(**extra):
    return ProjectedPPO(
        "MlpPolicy", gym.make("CartPole-v1"), seed=0, device="cpu", verbose=0,
        n_steps=64, batch_size=32, n_epochs=2,
        policy_kwargs={"net_arch": [16]}, **extra,
    )

probe = make_ppo()
names = projection_target_parameter_names(probe)   # actor params, in order
print("actor (projected) params:", names)

p0 = dict(probe.policy.named_parameters())
lower = [p0[n].detach() - 0.005 for n in names]
upper = [p0[n].detach() + 0.005 for n in names]

ppo = make_ppo(learning_rate=0.05, param_bounds_l=lower, param_bounds_u=upper)
critic_before = {n: p.detach().clone()
                 for n, p in ppo.policy.named_parameters() if "value_net" in n}
ppo.learn(total_timesteps=256)

final = dict(ppo.policy.named_parameters())
actor_inside = all(th.all(final[n] >= lo - 1e-6) and th.all(final[n] <= hi + 1e-6)
                   for n, lo, hi in zip(names, lower, upper))
critic_moved = any(not th.allclose(final[n], critic_before[n]) for n in critic_before)
print("actor stayed in box:", actor_inside, "| critic trained freely:", critic_moved)

actor (projected) params: ['action_net.bias', 'action_net.weight', 'mlp_extractor.policy_net.0.bias', 'mlp_extractor.policy_net.0.weight']
actor stayed in box: True | critic trained freely: True


## 4. Union of boxes and distance norms

The projection target can be a *union* of boxes — useful when a Rashomon
certificate yields several disjoint near-optimal regions. The parameters are
snapped into the **nearest** box, measured with the chosen `distance_norm`
(`"l2"`, `"l1"`, or `"linf"`). For a single box the norm is irrelevant
(projection is just a per-coordinate clamp).

Here the point `[10, 10]` is nearer (in every norm) to the second box `[8,9]^2`
than the first `[0,1]^2`, so it lands at `[9, 9]`.

In [5]:
x = th.nn.Parameter(th.tensor([10.0, 10.0]))
boxes_lower = [[th.tensor([0.0, 0.0])], [th.tensor([8.0, 8.0])]]   # far box, near box
boxes_upper = [[th.tensor([1.0, 1.0])], [th.tensor([9.0, 9.0])]]

n_clamped = project_to_interval_union([x], boxes_lower, boxes_upper, distance_norm="l2")
print("projected to nearest box:", x.detach().tolist(), "| coords clamped:", n_clamped)

projected to nearest box: [9.0, 9.0] | coords clamped: ProjectionResult(n_projected=2, n_boundary=2, selected_set_index=1, displacement_l2=1.4142135623730951, displacement_linf=1.0)


## 6. Projection diagnostics

Both `ProjectedDQN` and `ProjectedPPO` expose `projection_diagnostics()` (a
passthrough to the underlying `ProjectedAdam`). It answers the key questions:

* **How often was projection actually needed?** `projection_active_steps` and
  `projection_active_fraction` (steps where at least one weight was outside the
  bounds and had to be clamped, as a fraction of all bounded steps).
* **How much gets projected per update?** `mean_projected_elements_per_step`
  (over all steps) and `mean_projected_elements_per_active_step`; divide by
  `constrained_element_count` (also reported as `mean_projected_fraction_per_step`).
* **How hard is the constraint binding?** `mean/max_displacement_l2` and
  `..._linf` — the size of the projection step.
* **Pinned weights / sub-region:** `boundary_elements_total` and, for
  union-of-boxes, `selected_box_counts`.

During `learn()`, per-window rates are also written to the SB3 logger under the
`projection/` namespace (visible in TensorBoard). Use
`reset_projection_diagnostics()` to measure a fresh window.

In [6]:
import json

print("ProjectedDQN diagnostics:")
print(json.dumps(dqn.projection_diagnostics(), indent=2))
print("\nProjectedPPO diagnostics:")
print(json.dumps(ppo.projection_diagnostics(), indent=2))

ProjectedDQN diagnostics:
{
  "bounded_steps": 550,
  "projection_active_steps": 549,
  "projection_active_fraction": 0.9981818181818182,
  "projected_elements_total": 56994,
  "mean_projected_elements_per_step": 103.62545454545455,
  "mean_projected_elements_per_active_step": 103.81420765027322,
  "constrained_element_count": 114,
  "mean_projected_fraction_per_step": 0.9089952153110048,
  "mean_displacement_l2": 0.08656305700762465,
  "max_displacement_l2": 0.09508256228459677,
  "mean_displacement_linf": 0.011933990205553444,
  "max_displacement_linf": 0.013645108789205551,
  "boundary_elements_total": 56994,
  "mean_boundary_elements_per_step": 103.62545454545455,
  "selected_box_counts": {
    "0": 550
  }
}

ProjectedPPO diagnostics:
{
  "bounded_steps": 16,
  "projection_active_steps": 16,
  "projection_active_fraction": 1.0,
  "projected_elements_total": 1643,
  "mean_projected_elements_per_step": 102.6875,
  "mean_projected_elements_per_active_step": 102.6875,
  "constrained_e

## 5. Where bounds come from, and save/load

* **Trust region** (used above): a box of half-width $\varepsilon$ around the
  source weights — simple and certifies bounded parameter drift.
* **Rashomon certificate**: an `IntervalTrainer` produces per-parameter lower /
  upper bounds (single or union of boxes) describing all models that retain the
  source-task behaviour. Feed those tensors straight into `param_bounds_l/u`.

The escape hatch (no subclass) works for any SB3 algorithm: pass
`optimizer_class=ProjectedAdam` via `policy_kwargs`, then after construction call
`model.policy.optimizer.set_bounds(lower, upper, params=...)`.

**Save/Load:** projection bounds are *not* stored in the policy checkpoint. After
`load(...)`, re-attach them with `model.policy.optimizer.set_bounds(...)` to
resume constrained training.